In [1]:
import pandas as pd
import numpy as np


경로설정

In [2]:
PPMI_DIR = "./PPMI/"

UPSIT_FILE   = PPMI_DIR + "University_of_Pennsylvania_Smell_Identification_Test_UPSIT_12Apr2026.csv"
SCOPA_FILE   = PPMI_DIR + "SCOPA-AUT_12Apr2026.csv"
STATUS_FILE  = PPMI_DIR + "Participant_Status_12Apr2026.csv"

In [3]:
# ============================================================
# B-SIT 12항목 → SCENT 번호 매핑
# (원본 B-SIT의 Paint Thinner, Turpentine은 PPMI에 없어
#  Cherry, Dill Pickle로 대체)
# ============================================================
BSIT_MAP = {
    'CHERRY':      4,
    'DILL_PICKLE': 25,
    'BANANA':      7,
    'CHOCOLATE':   19,
    'CINNAMON':    15,
    'GASOLINE':    16,
    'LEMON':       36,
    'ONION':       11,
    'PINEAPPLE':   26,
    'ROSE':        39,
    'SOAP':        37,
    'SMOKE':       33,
}

UPSIT 로드 및 B-SIT 12항목 추출


In [4]:
upsit = pd.read_csv(UPSIT_FILE)

# SC(스크리닝) 우선, 없으면 BL(베이스라인) 사용
upsit = upsit[upsit['EVENT_ID'].isin(['SC', 'BL'])].copy()
upsit['EVENT_ORDER'] = upsit['EVENT_ID'].map({'SC': 0, 'BL': 1})
upsit = (upsit
         .sort_values(['PATNO', 'EVENT_ORDER'])
         .drop_duplicates(subset='PATNO', keep='first')
         .reset_index(drop=True))

# B-SIT 12항목 CORRECT 컬럼만 추출 후 이름 변경
correct_cols = {f'SCENT_{num:02d}_CORRECT': f'BSIT_{name}'
                for name, num in BSIT_MAP.items()}

upsit_sel = upsit[['PATNO', 'EVENT_ID'] + list(correct_cols.keys())].copy()
upsit_sel = upsit_sel.rename(columns={**correct_cols, 'EVENT_ID': 'UPSIT_EVENT_ID'})

# B-SIT 합산 점수 (0~12)
bsit_cols = [f'BSIT_{name}' for name in BSIT_MAP]
upsit_sel['BSIT_TOTAL'] = upsit_sel[bsit_cols].sum(axis=1)

print(f"[UPSIT] {len(upsit_sel)}명 추출 완료")

[UPSIT] 6127명 추출 완료



SCOPA-AUT 로드 및 변비 항목 추출 (SCAU5, SCAU6, SCAU7)

- SCAU5: 배변 횟수 감소 (0=없음 ~ 3=심함)
- SCAU6: 배변 시 힘주기 어려움
- SCAU7: 변실금(Fecal Incontinence)

In [5]:


scopa = pd.read_csv(SCOPA_FILE)

# BL(베이스라인) 우선, 없으면 가장 이른 방문 사용
event_order = {'SC':0,'BL':1,'V02':2,'V04':3,'V06':4,'V08':5,'V10':6,'V12':7}
scopa['EVENT_ORDER'] = scopa['EVENT_ID'].map(event_order).fillna(99)
scopa = (scopa
         .sort_values(['PATNO', 'EVENT_ORDER'])
         .drop_duplicates(subset='PATNO', keep='first')
         .reset_index(drop=True))

scopa_sel = scopa[['PATNO', 'EVENT_ID', 'SCAU5', 'SCAU6', 'SCAU7']].copy()
scopa_sel = scopa_sel.rename(columns={
    'EVENT_ID': 'SCOPA_EVENT_ID',
    'SCAU5': 'SCOPA_AUT5',
    'SCAU6': 'SCOPA_AUT6',
    'SCAU7': 'SCOPA_AUT7',
})

# 변비 합산 점수 (0~9)
scopa_sel['SCOPA_CONSTIPATION_TOTAL'] = scopa_sel[['SCOPA_AUT5','SCOPA_AUT6','SCOPA_AUT7']].sum(axis=1)

print(f"[SCOPA] {len(scopa_sel)}명 추출 완료")

[SCOPA] 4803명 추출 완료


Participant_Status 로드 - 진단 레이블

COHORT: 1=Parkinson's Disease, 2=Healthy Control\
        4=Prodromal (이 분석에서는 제외)


In [6]:

ps = pd.read_csv(STATUS_FILE)
ps_sel = ps[['PATNO', 'COHORT', 'COHORT_DEFINITION',]].copy()

print(f"[Status] {len(ps_sel)}명 로드 완료")

[Status] 8417명 로드 완료


데이터 병합 (PATNO 기준)

In [7]:
df = pd.merge(upsit_sel, scopa_sel, on='PATNO', how='inner')
df = pd.merge(df, ps_sel, on='PATNO', how='left')

# Prodromal(4), SWEDD(3) 제외 → PD(1)와 HC(2)만 사용
df = df[df['COHORT'].isin([1, 2])].copy()

# 레이블 생성: PD=1, HC=0
df['LABEL_PD'] = (df['COHORT'] == 1).astype(int)

print(f"\n[병합] PD+HC 총 {len(df)}명")
print(f"  PD (label=1): {(df['LABEL_PD']==1).sum()}명")
print(f"  HC (label=0): {(df['LABEL_PD']==0).sum()}명")


[병합] PD+HC 총 1012명
  PD (label=1): 868명
  HC (label=0): 144명


결측치 제거

In [8]:
feature_cols = bsit_cols + ['SCOPA_AUT5', 'SCOPA_AUT6', 'SCOPA_AUT7']

before = len(df)
df = df.dropna(subset=feature_cols).reset_index(drop=True)
after = len(df)

print(f"\n[결측치 제거] {before}명 → {after}명 ({before - after}명 제거)")
print(f"  PD (label=1): {(df['LABEL_PD']==1).sum()}명")
print(f"  HC (label=0): {(df['LABEL_PD']==0).sum()}명")


[결측치 제거] 1012명 → 944명 (68명 제거)
  PD (label=1): 809명
  HC (label=0): 135명


최종 데이터 확인 및 저장

In [ ]:
print(f"\n[최종 데이터셋]")
print(f"  크기: {df.shape}")
print(f"  컬럼: {list(df.columns)}")

# BSIT 값 자료형을 int로 변환 (0/1 이진값)
df[bsit_cols] = df[bsit_cols].astype(int)

df.to_csv(".\data\PPMI_BSIT12_SCOPA_최종.csv", index=False, encoding='utf-8-sig')
print("\n✅ 저장 완료: .\data\PPMI_BSIT12_SCOPA.csv")


[최종 데이터셋]
  크기: (944, 23)
  컬럼: ['PATNO', 'UPSIT_EVENT_ID', 'BSIT_CHERRY', 'BSIT_DILL_PICKLE', 'BSIT_BANANA', 'BSIT_CHOCOLATE', 'BSIT_CINNAMON', 'BSIT_GASOLINE', 'BSIT_LEMON', 'BSIT_ONION', 'BSIT_PINEAPPLE', 'BSIT_ROSE', 'BSIT_SOAP', 'BSIT_SMOKE', 'BSIT_TOTAL', 'SCOPA_EVENT_ID', 'SCOPA_AUT5', 'SCOPA_AUT6', 'SCOPA_AUT7', 'SCOPA_CONSTIPATION_TOTAL', 'COHORT', 'COHORT_DEFINITION', 'LABEL_PD']

✅ 저장 완료: .\data\PPMI_BSIT12_SCOPA_최종.csv
